# Customer Churn Prediction - Google Colab

This notebook runs the churn prediction task in Google Colab. Upload the dataset zip, train the models, and compare the results.

## 1. Upload Dataset Zip

In [ ]:
from google.colab import files
uploaded = files.upload()
zip_path = next(iter(uploaded.keys()))
zip_path

## 2. Extract Dataset

In [ ]:
from pathlib import Path
import zipfile

Path('data').mkdir(exist_ok=True)
with zipfile.ZipFile(zip_path) as z:
    for name in z.namelist():
        if name.endswith('Churn_Modelling.csv'):
            z.extract(name, 'data')

for file in Path('data').rglob('Churn_Modelling.csv'):
    target = Path('data') / 'Churn_Modelling.csv'
    if file != target:
        target.write_bytes(file.read_bytes())

list(Path('data').glob('*.csv'))

## 3. Train Models

In [ ]:
import joblib
import pandas as pd
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = pd.read_csv('data/Churn_Modelling.csv')
x = df.drop(columns=['RowNumber', 'CustomerId', 'Surname', 'Exited'])
y = df['Exited']

numeric_features = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary']
categorical_features = ['Geography', 'Gender']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

def preprocessor():
    return ColumnTransformer([
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_features),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_features),
    ])

models = {
    'logistic_regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'random_forest': RandomForestClassifier(n_estimators=180, max_depth=12, min_samples_leaf=8, class_weight='balanced', random_state=42),
    'gradient_boosting': GradientBoostingClassifier(n_estimators=160, learning_rate=0.05, max_depth=3, random_state=42),
}

rows = []
trained = {}
for name, clf in models.items():
    pipe = Pipeline([('preprocessor', preprocessor()), ('model', clf)])
    pipe.fit(x_train, y_train)
    pred = pipe.predict(x_test)
    prob = pipe.predict_proba(x_test)[:, 1]
    rows.append({
        'model': name,
        'accuracy': accuracy_score(y_test, pred),
        'precision': precision_score(y_test, pred, zero_division=0),
        'recall': recall_score(y_test, pred, zero_division=0),
        'f1_score': f1_score(y_test, pred, zero_division=0),
        'roc_auc': roc_auc_score(y_test, prob),
    })
    trained[name] = pipe

results = pd.DataFrame(rows).sort_values('roc_auc', ascending=False)
results

## 4. Save Best Model

In [ ]:
Path('models').mkdir(exist_ok=True)
best_model_name = results.iloc[0]['model']
joblib.dump({'model_name': best_model_name, 'model': trained[best_model_name]}, 'models/best_churn_model.joblib')
print('Best model:', best_model_name)